# Domino Detection & Classification Training

Two-stage pipeline:
1. **YOLO** — detects whole domino tiles in a photo (1 class: `domino`)
2. **Classifier** — reads pip count on each half (16 classes: `0-15`)

## Folder Structure
```
machine_learning/
  data/
    detector/          # YOLO training data (from Label Studio export)
      train/images/
      train/labels/
      val/images/
      val/labels/
    classifier/         # Pip classification data
      train/0-15/       # Sorted domino half images
      val/0-15/
      unsorted/         # Auto-cropped halves before sorting
  models/               # Trained model outputs
  notebooks/            # This notebook
  docs/                 # Research notes
```

## Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install ultralytics tensorflow pillow matplotlib

In [ ]:
import os
import numpy as np
from pathlib import Path

# Project paths
ML_ROOT = Path('../').resolve()
DATA_DIR = ML_ROOT / 'data'
DETECTOR_DIR = DATA_DIR / 'detector'
CLASSIFIER_DIR = DATA_DIR / 'classifier'
MODELS_DIR = ML_ROOT / 'models'

print(f'ML Root: {ML_ROOT}')
print(f'Detector data: {DETECTOR_DIR}')
print(f'Classifier data: {CLASSIFIER_DIR}')
print(f'Models output: {MODELS_DIR}')

---
## Stage 1: YOLO Domino Detection

### 1.1 Import Label Studio Export

After labeling in Label Studio:
1. Export as **YOLO** format
2. Unzip into `data/detector/`
3. Split into `train/` and `val/` (80/20)

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

def split_label_studio_export(export_dir, output_dir, val_split=0.2):
    """Split a Label Studio YOLO export into train/val sets."""
    export_dir = Path(export_dir)
    output_dir = Path(output_dir)
    
    images_dir = export_dir / 'images'
    labels_dir = export_dir / 'labels'
    
    # Get all image files
    image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))
    print(f'Found {len(image_files)} images')
    
    # Split
    train_imgs, val_imgs = train_test_split(image_files, test_size=val_split, random_state=42)
    print(f'Train: {len(train_imgs)}, Val: {len(val_imgs)}')
    
    # Copy files
    for split_name, split_imgs in [('train', train_imgs), ('val', val_imgs)]:
        img_out = output_dir / split_name / 'images'
        lbl_out = output_dir / split_name / 'labels'
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        
        for img_path in split_imgs:
            # Copy image
            shutil.copy2(img_path, img_out / img_path.name)
            # Copy corresponding label
            label_path = labels_dir / f'{img_path.stem}.txt'
            if label_path.exists():
                shutil.copy2(label_path, lbl_out / label_path.name)
    
    print('Done!')

# Usage: update the path to your Label Studio export
# split_label_studio_export('path/to/label-studio-export', DETECTOR_DIR)

### 1.2 Create YOLO data config

In [ ]:
data_yaml_content = f"""path: {DETECTOR_DIR}
train: train/images
val: val/images

nc: 1
names: ['domino']
"""

data_yaml_path = DETECTOR_DIR / 'data.yaml'
data_yaml_path.write_text(data_yaml_content)
print(f'Wrote {data_yaml_path}')
print(data_yaml_content)

### 1.3 Train YOLO

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv11-nano
model = YOLO('yolo11n.pt')

# Train
results = model.train(
    data=str(DETECTOR_DIR / 'data.yaml'),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(MODELS_DIR),
    name='domino_detector'
)

### 1.4 Validate YOLO

In [ ]:
# Validate on the val set
metrics = model.val()
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

---
## Stage 2: Pip Classifier

### 2.1 Auto-crop domino halves from YOLO detections

In [ ]:
from PIL import Image

def crop_and_split_detections(model, source_dir, output_dir):
    """Run YOLO on images, crop detections, split each in half."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    results = model.predict(source=str(source_dir), conf=0.5)
    
    count = 0
    for result in results:
        img = Image.open(result.path)
        img_name = Path(result.path).stem
        
        for i, box in enumerate(result.boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            crop = img.crop((x1, y1, x2, y2))
            w, h = crop.size
            
            if h > w:
                # Vertical — split top/bottom
                half_a = crop.crop((0, 0, w, h // 2))
                half_b = crop.crop((0, h // 2, w, h))
            else:
                # Horizontal — split left/right
                half_a = crop.crop((0, 0, w // 2, h))
                half_b = crop.crop((w // 2, 0, w, h))
            
            half_a.save(output_dir / f'{img_name}_d{i}_a.jpg')
            half_b.save(output_dir / f'{img_name}_d{i}_b.jpg')
            count += 1
    
    print(f'Cropped {count} dominoes into {count * 2} halves')
    print(f'Saved to {output_dir}')
    print(f'Next step: manually sort these into folders 0/ through 15/')

# Usage: point at a folder of domino photos
# best_yolo = YOLO(str(MODELS_DIR / 'domino_detector' / 'weights' / 'best.pt'))
# crop_and_split_detections(best_yolo, 'path/to/photos', CLASSIFIER_DIR / 'unsorted')

### 2.2 Create classifier folder structure

After sorting the `unsorted/` halves into pip folders, split into train/val.

In [ ]:
def create_classifier_folders():
    """Create 0-15 folders for train and val."""
    for split in ['train', 'val']:
        for pip_count in range(16):
            folder = CLASSIFIER_DIR / split / str(pip_count)
            folder.mkdir(parents=True, exist_ok=True)
    print('Created classifier folders: train/0-15 and val/0-15')

create_classifier_folders()

In [ ]:
def split_classifier_data(sorted_dir, train_dir, val_dir, val_split=0.2):
    """Split manually sorted pip images into train/val.
    
    Expects sorted_dir to contain folders 0/ through 15/ with images.
    """
    sorted_dir = Path(sorted_dir)
    train_dir = Path(train_dir)
    val_dir = Path(val_dir)
    
    for pip_folder in sorted(sorted_dir.iterdir()):
        if not pip_folder.is_dir():
            continue
        
        images = list(pip_folder.glob('*.jpg')) + list(pip_folder.glob('*.png'))
        if not images:
            continue
        
        train_imgs, val_imgs = train_test_split(images, test_size=val_split, random_state=42)
        
        for img in train_imgs:
            shutil.copy2(img, train_dir / pip_folder.name / img.name)
        for img in val_imgs:
            shutil.copy2(img, val_dir / pip_folder.name / img.name)
        
        print(f'  Pip {pip_folder.name}: {len(train_imgs)} train, {len(val_imgs)} val')

# Usage: after manually sorting unsorted/ into folders 0-15
# split_classifier_data(
#     CLASSIFIER_DIR / 'sorted',
#     CLASSIFIER_DIR / 'train',
#     CLASSIFIER_DIR / 'val'
# )

### 2.3 Train Pip Classifier

In [ ]:
import tensorflow as tf

IMG_SIZE = (100, 100)
BATCH_SIZE = 32
NUM_CLASSES = 16  # 0-15 pips for double-15 set

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(CLASSIFIER_DIR / 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(CLASSIFIER_DIR / 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Print class names to verify order
print('Classes:', train_ds.class_names)

In [ ]:
# Build classifier model
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(100, 100, 3)),
    
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train with checkpointing
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    str(MODELS_DIR / 'pip_classifier_best.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[checkpoint]
)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

---
## Export Models for Mobile

In [ ]:
# Export YOLO to TFLite
best_yolo = YOLO(str(MODELS_DIR / 'domino_detector' / 'weights' / 'best.pt'))
best_yolo.export(format='tflite', int8=True, imgsz=640)
print('YOLO exported to TFLite')

In [ ]:
# Export pip classifier to TFLite
classifier = tf.keras.models.load_model(str(MODELS_DIR / 'pip_classifier_best.h5'))

converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

output_path = MODELS_DIR / 'pip_classifier.tflite'
with open(output_path, 'wb') as f:
    f.write(tflite_model)

print(f'Classifier exported to {output_path}')
print(f'Size: {len(tflite_model) / 1024:.1f} KB')

In [ ]:
# Generate labels file
labels_path = MODELS_DIR / 'labels_classifier.txt'
labels_path.write_text('\n'.join(str(i) for i in range(16)))
print(f'Labels written to {labels_path}')